# Sandbox

A scratch notebook: somewhere to try a query without disturbing 01, 02 or 03. §4 is the reason it exists today — the playlist comparison Marc asked for — and §5 is deliberately empty and meant to stay that way in git.

It reads the warehouse and makes no Spotify API call. The profile comes from `SPOT_PROFILE`, so `make report-05 PROFILE=<slug>` chooses the person and this notebook names nobody.

## 1. Connect

`setup()` resolves the profile, opens the connection and configures the plotting theme. It is the only cell that knows how to reach the database.

In [ ]:
from IPython.display import Markdown

from spotify_lakehouse import playlists
from spotify_lakehouse.notebook import setup

ctx = setup()
display(
    Markdown(
        f"Profile **`{ctx.profile}`** · session **`{ctx.session}`** · "
        f"schemas `{ctx.stg_schema}` and `{ctx.mart_schema}`."
    )
)

## 2. The API, in one screen

Four ways to ask the warehouse a question. **It is `ctx.conn`**, not `ctx.con`.

| Call | Returns | One-liner |
|---|---|---|
| `ctx.frame(sql, params)` | `pandas.DataFrame` | `ctx.frame("select * from {mart}.dim_playlist limit 5")` |
| `ctx.rows(sql, params)` | list of tuples | `ctx.rows("select count(*) from {mart}.fct_play_event")` |
| `ctx.scalar(sql, params)` | one value | `ctx.scalar("select count(*) from {mart}.dim_content")` |
| `ctx.conn` | the psycopg connection | `with ctx.conn.cursor() as cur: ...` |

`{stg}` and `{mart}` are replaced with this session's schema names, so a query written here runs unchanged in a worktree. Double any literal brace.

In [ ]:
example = ctx.frame(
    "select playlist_name, track_total from {mart}.dim_playlist "
    "where is_current and playlist_key > 0 order by track_total desc limit 5"
)
display(example)
display(
    Markdown(
        f"`dim_content` holds **{ctx.scalar('select count(*) from {mart}.dim_content'):,}** rows; "
        f"`ctx.conn` is a `{type(ctx.conn).__name__}`."
    )
)

## 3. What you can query

Every table, its row count and a runnable example live in **notebook 01 §11, The Queryable Catalog** — introspected at run time, so it cannot go stale. This section deliberately does not copy it: a second list would drift the first time a model landed.

`make report-01` renders it.

## 4. Playlist comparison

The question: **which songs are in one playlist and not in the other.** `miss` is directional (data-contracts §5), so every answer below says which direction it is measuring.

### 4.1 The playlists that were loaded

⚠️ There is no *owner* column. A playlist's owner is a person who is not necessarily Marc, and this repository is public, so owner identity is discarded before anything is stored (R-054). "Is this mine?" is therefore not answerable from the warehouse.

In [ ]:
catalog = playlists.load_playlists(ctx)
display(catalog.head(25))
display({"playlists loaded": len(catalog), "tracks loaded": int(catalog["tracks_loaded"].sum())})

### 4.2 Resolving the two names

Resolved case-insensitively, and **never guessed**: a name matching zero playlists or more than one prints the candidates and stops, because silently picking one answers a different question than the one asked.

In [ ]:
# Marc's own words, 2026-09-16. Matching folds apostrophe variants, because Spotify stores
# `Connor\u2019s Playlist` with a typographic apostrophe and an ASCII one finds nothing.
LEFT_NAME, RIGHT_NAME = "Smith", "Connor's Playlist"
membership = playlists.load_membership(ctx)

# Resolve against EVERY playlist, not only those with rows loaded. A playlist Spotify refused has no
# membership, so resolving against the membership frame makes it invisible — and a leftover
# incidental match then looks unique and answers a different question (R-054).
matches, problems = {}, []
for needle in (LEFT_NAME, RIGHT_NAME):
    try:
        matches[needle] = playlists.resolve(catalog, needle)
    except playlists.AmbiguousPlaylist as exc:
        problems.append(str(exc))

for needle, match in matches.items():
    loaded = int(catalog.loc[catalog["playlist_name"] == match.name, "tracks_loaded"].sum())
    how = "exact name" if match.exact else "substring match"
    display(Markdown(f"**{needle}** → **{match.name}** ({how}, {loaded:,} tracks loaded)"))
    if match.warning:
        display(Markdown(f"> [!WARNING]\n> {match.warning}"))
    if loaded == 0:
        problems.append(
            f"{match.name!r} has no tracks loaded, so it cannot be compared. Spotify refused it "
            "(403) — see the load report."
        )

for problem in problems:
    display(Markdown(f"> [!WARNING]\n> {problem}"))

shown = sorted(
    set(playlists.candidates(catalog, LEFT_NAME))
    | set(playlists.candidates(catalog, RIGHT_NAME))
    | set(playlists.candidates(catalog, "Connor"))
)
display(Markdown("**Every playlist whose name contains either word:**"))
display(
    catalog[catalog["playlist_name"].isin(shown)][["playlist_name", "track_total", "tracks_loaded"]]
)
ready = len(matches) == 2 and not problems

### 4.3 The answer

<!-- caption: Tracks present in the first playlist and absent from the second, by artist and title -->

In [ ]:
if ready:
    left_name, right_name = matches[LEFT_NAME].name, matches[RIGHT_NAME].name
    left = membership[membership["playlist_name"] == left_name]
    right = membership[membership["playlist_name"] == right_name]
    answer = playlists.misses(
        left, right, left_name=left_name, right_name=right_name, tier="content_uri"
    )
    display(
        Markdown(
            f"### {answer.label}\n\n"
            f"**{answer.miss_count}** of **{answer.left_size}** tracks. "
            f"{answer.coverage_note}."
        )
    )
    display(answer.rows[["artist", "title"]])
else:
    display(
        Markdown(
            "> [!WARNING]\n> Both names must resolve to exactly one playlist "
            "before §4.3 can run. See §4.2."
        )
    )

### 4.4 All three tiers, side by side

data-contracts §5: *the gap between tiers is itself the story* — tier 1 to tier 2 is re-issue churn, tier 2 to tier 3 is genuine ambiguity.

⚠️ **Tier 2's reach depends on which side you ask about.** Across `dim_content` as a whole, ISRC lands only on API-resolved tracks — a little over 1% of rows. But a *playlist* payload carries `external_ids.isrc` on the track object itself, so for playlist membership tier 2 covers almost everything. The `tier_coverage` column prints the share for the playlist in hand rather than assuming either figure, because the unqualified sentence "tier 2 only covers API-resolved tracks" is true of the warehouse and misleading about this comparison.

In [ ]:
if ready:
    display(playlists.tier_table(left, right, left_name=left_name, right_name=right_name))
else:
    display(Markdown("> [!WARNING]\n> Not run: see §4.2."))

### 4.5 The other direction

Marc asked for one direction. The reverse costs nothing and is the check that the first answer is not simply an artefact of one playlist being larger than the other.

In [ ]:
if ready:
    reverse = playlists.misses(
        right, left, left_name=right_name, right_name=left_name, tier="content_uri"
    )
    display(
        Markdown(
            f"### {reverse.label}\n\n**{reverse.miss_count}** of **{reverse.left_size}** tracks."
        )
    )
    display(reverse.rows[["artist", "title"]])
    display(playlists.tier_table(right, left, left_name=right_name, right_name=left_name))
else:
    display(Markdown("> [!WARNING]\n> Not run: see §4.2."))

### 4.6 How much of this the warehouse already knew

A playlist can contain tracks nobody has played, and `dim_content` is built from plays and API track lookups. This is the share of playlist tracks that already had a row.

In [ ]:
display(playlists.orphan_summary(ctx))
display(membership.groupby("in_dim_content").size().rename("tracks").reset_index())

## 5. Scratch

Empty on purpose. Add a cell, ask the warehouse something, and keep or discard it — a diff of this notebook should show only what you added.

The warehouse connection is closed at the end of the run.

In [ ]:
ctx.close()